# Second-order cone programs with `pounce.qp.solve_socp`

A **second-order (Lorentz) cone** is
$$\mathcal Q^m = \{\, (t, u) \in \mathbb R\times\mathbb R^{m-1} : t \ge \|u\|_2 \,\}.$$
An SOCP minimizes a linear/quadratic objective subject to equalities and a
product of cones — nonnegative orthants *and* second-order cones:
$$\min_x\;\tfrac12 x^\top P x + c^\top x \quad\text{s.t.}\quad A x = b,\;\; G x \preceq_{\mathcal K} h.$$

POUNCE solves this with the same Mehrotra predictor–corrector machinery as
the LP/QP path, now with **Nesterov–Todd scaling** for the cones. The Python
call mirrors `solve_qp` but adds a `cones=` partition of the rows of `G`:
each slack block $s = h - Gx$ must lie in its cone.

> **Inspiration.** The conic design follows
> [Clarabel](https://github.com/oxfordcontrol/Clarabel.rs) (Goulart & Chen);
> the sparse second-order-cone KKT representation follows
> [ECOS](https://github.com/embotech/ecos) (Domahidi, Chu & Boyd). POUNCE is
> pure Rust and wraps neither.

In [1]:
import numpy as np
from pounce.qp import solve_socp

np.set_printoptions(precision=4, suppress=True)

## 1. Norm minimization (projection)

Minimize $\|x - a\|$ — i.e. find the closest point to $a$ inside whatever
feasible set we impose. With **no** other constraint the answer is trivially
$x = a$, which makes it a perfect first sanity check of the machinery.

Epigraph form: introduce $t$ and minimize $t$ subject to $(t, x-a)\in\mathcal Q$:
- variable vector is $(t, x_0, x_1)$,
- slack $s = h - Gx$ must equal $(t,\; x_0-a_0,\; x_1-a_1)$.

Take $a = (2, -1)$. With $G = -I$ and $h = (0, -2, 1)$ we get
$s = (t,\, x_0 - 2,\, x_1 + 1)$. The optimum is $t^\* = 0,\ x = (2, -1)$.

In [2]:
a = np.array([2.0, -1.0])
r = solve_socp(
    c=[1.0, 0.0, 0.0],                 # minimize t   (decision vars: t, x0, x1)
    G=-np.eye(3), h=[0.0, -a[0], -a[1]],
    cones=[("soc", 3)],
)
t, x = r.x[0], r.x[1:]
print(r.status, "  t =", round(t, 6), "  x =", x)
assert r.success and abs(t) < 1e-6 and np.allclose(x, a, atol=1e-6)

optimal   t = 0.0   x = [ 2. -1.]


## 2. Linear objective over a ball — a closed-form check

$$\min_x\; c^\top x \quad\text{s.t.}\quad \|x - a\|_2 \le r.$$
The minimizer pushes from $a$ straight down $-c$ to the ball boundary:
$$x^\* = a - r\,\frac{c}{\|c\|},\qquad \text{obj}^\* = c^\top a - r\,\|c\|.$$

The cone constraint $\|x-a\|\le r$ becomes $(r,\ x-a)\in\mathcal Q$:
slack row 0 is the constant $r$ (so $G$ row 0 is zero, $h_0=r$), and the
remaining rows give $s_i = a_i - x_i$ (so $G = I$, $h_i = a_i$).

In [3]:
n = 4
rng = np.random.default_rng(1)
a = rng.standard_normal(n)
c = rng.standard_normal(n)
r_ball = 0.7

G = np.vstack([np.zeros((1, n)), np.eye(n)])   # (n+1) x n
h = np.concatenate([[r_ball], a])
res = solve_socp(c=c, G=G, h=h, cones=[("soc", n + 1)])

x_star = a - r_ball * c / np.linalg.norm(c)
print("status   :", res.status)
print("x        :", res.x)
print("closed   :", x_star)
print("obj      :", res.obj, " vs closed", float(c @ a - r_ball * np.linalg.norm(c)))
print("on bdry  : ||x-a|| =", np.linalg.norm(res.x - a), " (= r =", r_ball, ")")
assert res.success and np.allclose(res.x, x_star, atol=1e-6)

status   : optimal
x        : [-0.1485  0.578   0.6235 -1.6203]
closed   : [-0.1485  0.578   0.6235 -1.6203]
obj      : -1.1528769457423376  vs closed -1.1528769477540548
on bdry  : ||x-a|| = 0.6999999984314715  (= r = 0.7 )


## 3. Constrained least squares (SOCP epigraph of a 2-norm)

$$\min_x\; \|Mx - d\|_2 \quad\text{s.t.}\quad \mathbf 1^\top x = 1.$$
Epigraph: minimize $t$ with $(t,\ Mx - d)\in\mathcal Q$ and the equality
$\mathbf 1^\top x = 1$. We compare against the analytic equality-constrained
least-squares solution (a KKT linear system).

In [4]:
rng = np.random.default_rng(2)
m, n = 6, 3
M = rng.standard_normal((m, n))
d = rng.standard_normal(m)

# decision vars: (t, x_0..x_{n-1}); slack s = (t, d - M x) in SOC(m+1).
nv = 1 + n
c = np.zeros(nv); c[0] = 1.0
G = np.zeros((m + 1, nv))
G[0, 0] = -1.0                 # s_0 = t
G[1:, 1:] = M                  # s_i = d_i - (M x)_i
h = np.concatenate([[0.0], d])
A = np.concatenate([[0.0], np.ones(n)])[None, :]   # sum(x) = 1
res = solve_socp(c=c, G=G, h=h, A=A, b=[1.0], cones=[("soc", m + 1)])

# Analytic equality-constrained least squares via the normal-equation KKT.
MtM = M.T @ M
KKT = np.block([[MtM, np.ones((n, 1))], [np.ones((1, n)), np.zeros((1, 1))]])
rhs = np.concatenate([M.T @ d, [1.0]])
x_ref = np.linalg.solve(KKT, rhs)[:n]
print("status :", res.status)
print("x      :", res.x[1:])
print("ref    :", x_ref)
print("t = ||Mx-d|| :", res.x[0], " vs", np.linalg.norm(M @ x_ref - d))
assert res.success and np.allclose(res.x[1:], x_ref, atol=1e-6)

status : optimal
x      : [0.5684 0.3224 0.1092]
ref    : [0.5684 0.3224 0.1092]
t = ||Mx-d|| : 1.1640358950466765  vs 1.1640359009413206


## 4. Mixed cones

Cones **compose**: a `cones=[("nonneg", k), ("soc", m)]` partition puts the
first $k$ slacks in $\mathbb R^k_+$ and the next $m$ in a second-order cone.

Here we minimize $-x_0 - x_1$ over the unit ball $\|x\|\le 1$ (a 3-row SOC
slack $(1, x_0, x_1)$) *and* the linear cut $x_1 \le 0.5$ (a 1-row nonnegative
slack). We verify feasibility and KKT stationarity from the returned duals.

In [5]:
# rows: [ nonneg: 0.5 - x1 >= 0 ] then [ soc: (1, x0, x1) ]
G = np.array([
    [0.0,  1.0],   # nonneg slack s0 = 0.5 - x1
    [0.0,  0.0],   # soc s0 = 1            (constant)
    [-1.0, 0.0],   # soc s1 = x0
    [0.0, -1.0],   # soc s2 = x1
])
h = np.array([0.5, 1.0, 0.0, 0.0])
c = np.array([-1.0, -1.0])
res = solve_socp(c=c, G=G, h=h, cones=[("nonneg", 1), ("soc", 3)])

x = res.x
s = h - G @ x
print("status :", res.status, "  x =", x)
print("nonneg slack (>=0)      :", s[0])
print("soc slack (t>=||u||)    :", s[1], ">=", np.linalg.norm(s[2:]))
# KKT stationarity:  c + G^T z = 0   (no P, no A here)
print("stationarity ||c+G^T z||:", np.linalg.norm(c + G.T @ res.z))
assert res.success and s[0] > -1e-7 and s[1] + 1e-7 >= np.linalg.norm(s[2:])
assert np.linalg.norm(c + G.T @ res.z) < 1e-6

status : optimal   x = [0.866 0.5  ]
nonneg slack (>=0)      : 1.8324023409732604e-09
soc slack (t>=||u||)    : 1.0 >= 0.9999999998753488
stationarity ||c+G^T z||: 2.544536473493165e-09


## 5. A larger cone (sparse KKT)

Large second-order cones use a **diagonal-plus-rank-1** KKT representation —
one auxiliary variable per cone (the ECOS/Clarabel "sparse SOC" trick) — so
the factorization stays sparse instead of dropping a dense $m\times m$ block.
We solve the ball problem of §2 at dimension $n = 50$ and confirm the same
closed form.

In [6]:
n = 50
rng = np.random.default_rng(3)
a = rng.standard_normal(n)
c = rng.standard_normal(n)
r_ball = 1.3

G = np.vstack([np.zeros((1, n)), np.eye(n)])
h = np.concatenate([[r_ball], a])
res = solve_socp(c=c, G=G, h=h, cones=[("soc", n + 1)])

x_star = a - r_ball * c / np.linalg.norm(c)
print("status :", res.status, "  iters:", res.iters)
print("max |x - closed form| :", np.max(np.abs(res.x - x_star)))
print("obj  :", res.obj, " vs closed", float(c @ a - r_ball * np.linalg.norm(c)))
assert res.success and np.allclose(res.x, x_star, atol=1e-5)

status : optimal   iters: 8
max |x - closed form| : 7.617580344287944e-09
obj  : -3.315024725095695  vs closed -3.315024727917102


## Where next

- **`15_differentiable_convex.ipynb`** — differentiate these SOCP solutions
  w.r.t. their data $P, c, G, h, A, b$ with `pounce.jax.solve_socp`.
- The [Convex Solver chapter](../../docs/src/convex-solver.md) covers the
  cone abstraction, warm starting, and the sparse-KKT representation in
  more detail.